# AI-XRay · 03 — Entrenamiento + registro completo en MLflow

**Proyecto Final Independiente** · César Libardo España Salguero

### Qué se entrena aquí

| # | Run | Tipo | Cambio respecto al anterior | Hipótesis |
|---|---|---|---|---|
| 1 | `ref1_frozen_lr1e-3_do0.3` | Referencia | — | Línea base: solo la cabeza nueva |
| 2 | `ref2_frozen_lr1e-4_do0.5` | Referencia | LR ×0.1, dropout 0.5 | Más regularizado, aprende más lento |
| 3 | `ref3_finetune_1paso_lr1e-5` | Referencia | Base descongelada desde la capa 143, cabeza **aleatoria** | Adapta los filtros a rayos X |
| A | `varA_finetune_2fases_lr1e-5` | **Propia** | Fine-tuning que **parte del modelo del Run 1** | Ver justificación abajo |
| B | `varB_frozen_sin_flip` | **Propia** | Run 1 **sin flip horizontal** (único cambio) | Ver justificación abajo |

**Criterios de comparación justa (cambio respecto a la versión de clase):**
- Todos los runs tienen el **mismo presupuesto**: `EPOCHS = 10` con `EarlyStopping(patience=4)`.
  En clase se usaban 3 épocas, y el Run 2 seguía mejorando en la última: se comparaba un modelo
  subentrenado contra otros que ya habían convergido.
- Cada run se evalúa en **validación** (que es lo que se usa para elegir) y en **test** (que solo
  se usa para reportar). Las métricas de test no intervienen en ninguna decisión.
- En cada run se calcula también el **umbral clínico**: el umbral más alto que, en validación,
  detecta ≥95 % de las neumonías. Se aplica tal cual al test (métricas `clin_test_*`).

In [1]:
%matplotlib inline
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import mlflow

from src.config import MLFLOW_TRACKING_URI, SPLIT_MANIFEST_PATH, setup_mlflow
from src.preprocessing import dataset_from_manifest
from src.train import compute_balanced_class_weight, train_and_log

split = pd.read_csv(SPLIT_MANIFEST_PATH)
train_ds = dataset_from_manifest(split, "train")
train_ds_sin_flip = dataset_from_manifest(split, "train", horizontal_flip=False)
val_ds = dataset_from_manifest(split, "val")
test_ds = dataset_from_manifest(split, "test")

class_weight = compute_balanced_class_weight(split[split["split"] == "train"]["label"])
print("class_weight:", {k: round(v, 4) for k, v in class_weight.items()})

experiment = setup_mlflow()
print("Tracking URI:", MLFLOW_TRACKING_URI.replace(str(PROJECT_ROOT.as_posix()), "<AI-XRay>"))
print("Experiment:", experiment.name, "| artefactos en:", Path(experiment.artifact_location.replace("file://", "")).name)

class_weight: {0: 0.9907, 1: 1.0094}


2026/10/08 14:04:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/08 14:04:21 INFO mlflow.store.db.utils: Updating database tables


Tracking URI: sqlite:///<AI-XRay>/mlflow.db
Experiment: AI-XRay | artefactos en: mlruns


In [2]:
EPOCHS = 10        # mismo presupuesto para TODOS los runs
FINE_TUNE_AT = 143 # ResNet50 tiene 175 capas: se descongela el último bloque (conv5)
resultados = {}

## Run 1 (referencia) — ResNet50 congelado, LR=1e-3, dropout=0.3

In [3]:
run_id_1, model_1, resultados["ref1"] = train_and_log(
    train_ds, val_ds, test_ds,
    run_name="ref1_frozen_lr1e-3_do0.3",
    learning_rate=1e-3, dropout=0.3, epochs=EPOCHS,
    fine_tune=False, class_weight=class_weight,
    tags={"fase": "A_congelado", "tipo": "referencia"},
    justification="Run de referencia de clase: línea base con la base de ResNet50 congelada.",
)

Epoch 1/10


44/44 - 129s - 3s/step - accuracy: 0.8099 - auc: 0.8949 - loss: 0.4084 - val_accuracy: 0.9159 - val_auc: 0.9642 - val_loss: 0.2523 - learning_rate: 0.0010


Epoch 2/10


44/44 - 143s - 3s/step - accuracy: 0.9107 - auc: 0.9620 - loss: 0.2470 - val_accuracy: 0.9094 - val_auc: 0.9713 - val_loss: 0.2211 - learning_rate: 0.0010


Epoch 3/10


...

44/44 - 160s - 4s/step - accuracy: 0.9402 - auc: 0.9833 - loss: 0.1635 - val_accuracy: 0.9320 - val_auc: 0.9809 - val_loss: 0.1819 - learning_rate: 0.0010


Epoch 10/10


44/44 - 119s - 3s/step - accuracy: 0.9316 - auc: 0.9810 - loss: 0.1752 - val_accuracy: 0.9353 - val_auc: 0.9825 - val_loss: 0.1722 - learning_rate: 0.0010


=== VALIDACIÓN (umbral 0.50) ===
              precision    recall  f1-score   support

      NORMAL      0.946     0.922     0.934       153
   PNEUMONIA      0.925     0.949     0.937       156

    accuracy                          0.935       309
   macro avg      0.936     0.935     0.935       309
weighted avg      0.936     0.935     0.935       309

=== TEST (umbral 0.50) ===
              precision    recall  f1-score   support

      NORMAL      0.905     0.918     0.912       146
   PNEUMONIA      0.922     0.910     0.916       156

    accuracy                          0.914       302
   macro avg      0.914     0.914     0.914       302
weighted avg      0.914     0.914     0.914       302

=== TEST (umbral clínico 0.4287, fijado con validación para recall>=0.95) ===
              precision    recall  f1-score   support

      NORMAL      0.923     0.897     0.910       146
   PNEUMONIA      0.906     0.929     0.918       156

    accuracy                          0.914 

Run 'ref1_frozen_lr1e-3_do0.3' -- run_id=a4bf9a5c2e6b454d8e34a022562e6aef -- 10 épocas, 21.8 min
  VAL  @0.5: {"accuracy": 0.9353, "precision": 0.925, "recall": 0.9487, "specificity": 0.9216, "f1_score": 0.9367, "auc": 0.9823, "tp": 148, "fp": 12, "tn": 141, "fn": 8}
  TEST @0.5: {"accuracy": 0.9139, "precision": 0.9221, "recall": 0.9103, "specificity": 0.9178, "f1_score": 0.9161, "auc": 0.9792, "tp": 142, "fp": 12, "tn": 134, "fn": 14}


## Run 2 (referencia) — ResNet50 congelado, LR=1e-4, dropout=0.5

In [4]:
run_id_2, _, resultados["ref2"] = train_and_log(
    train_ds, val_ds, test_ds,
    run_name="ref2_frozen_lr1e-4_do0.5",
    learning_rate=1e-4, dropout=0.5, epochs=EPOCHS,
    fine_tune=False, class_weight=class_weight,
    tags={"fase": "A_congelado", "tipo": "referencia"},
    justification="Run de referencia de clase: más regularización (dropout 0.5) y LR 10 veces menor.",
)

Epoch 1/10


44/44 - 129s - 3s/step - accuracy: 0.6076 - auc: 0.6544 - loss: 0.7510 - val_accuracy: 0.7411 - val_auc: 0.8762 - val_loss: 0.5067 - learning_rate: 1.0000e-04


Epoch 2/10


44/44 - 166s - 4s/step - accuracy: 0.6659 - auc: 0.7367 - loss: 0.6393 - val_accuracy: 0.8091 - val_auc: 0.9301 - val_loss: 0.4279 - learning_rate: 1.0000e-04


Epoch 3/10


...

44/44 - 119s - 3s/step - accuracy: 0.8380 - auc: 0.9162 - loss: 0.3697 - val_accuracy: 0.9094 - val_auc: 0.9656 - val_loss: 0.2541 - learning_rate: 1.0000e-04


Epoch 10/10


44/44 - 119s - 3s/step - accuracy: 0.8719 - auc: 0.9369 - loss: 0.3220 - val_accuracy: 0.9094 - val_auc: 0.9670 - val_loss: 0.2457 - learning_rate: 1.0000e-04


=== VALIDACIÓN (umbral 0.50) ===
              precision    recall  f1-score   support

      NORMAL      0.898     0.922     0.910       153
   PNEUMONIA      0.921     0.897     0.909       156

    accuracy                          0.909       309
   macro avg      0.910     0.910     0.909       309
weighted avg      0.910     0.909     0.909       309

=== TEST (umbral 0.50) ===
              precision    recall  f1-score   support

      NORMAL      0.868     0.904     0.886       146
   PNEUMONIA      0.907     0.872     0.889       156

    accuracy                          0.887       302
   macro avg      0.888     0.888     0.887       302
weighted avg      0.888     0.887     0.887       302

=== TEST (umbral clínico 0.2479, fijado con validación para recall>=0.95) ===
              precision    recall  f1-score   support

      NORMAL      0.922     0.733     0.817       146
   PNEUMONIA      0.790     0.942     0.860       156

    accuracy                          0.841 

Run 'ref2_frozen_lr1e-4_do0.5' -- run_id=284800fdc5ed41ad8769128820a038ed -- 10 épocas, 23.6 min
  VAL  @0.5: {"accuracy": 0.9094, "precision": 0.9211, "recall": 0.8974, "specificity": 0.9216, "f1_score": 0.9091, "auc": 0.9669, "tp": 140, "fp": 12, "tn": 141, "fn": 16}
  TEST @0.5: {"accuracy": 0.8874, "precision": 0.9067, "recall": 0.8718, "specificity": 0.9041, "f1_score": 0.8889, "auc": 0.9632, "tp": 136, "fp": 14, "tn": 132, "fn": 20}


## Run 3 (referencia) — Fine-tuning "en un solo paso", LR=1e-5

Tal como en clase: se construye un modelo **nuevo**, con la cabeza inicializada al azar y la base
descongelada desde la capa 143, y se entrena todo junto con un LR muy bajo.

In [5]:
run_id_3, _, resultados["ref3"] = train_and_log(
    train_ds, val_ds, test_ds,
    run_name="ref3_finetune_1paso_lr1e-5",
    learning_rate=1e-5, dropout=0.3, epochs=EPOCHS,
    fine_tune=True, fine_tune_at=FINE_TUNE_AT, class_weight=class_weight,
    tags={"fase": "B_fine_tuning", "tipo": "referencia"},
    justification="Run de referencia de clase: fine-tuning con cabeza aleatoria y LR 1e-5.",
)

Epoch 1/10


44/44 - 175s - 4s/step - accuracy: 0.7862 - auc: 0.8885 - loss: 0.4586 - val_accuracy: 0.8803 - val_auc: 0.9709 - val_loss: 0.3053 - learning_rate: 1.0000e-05


Epoch 2/10


44/44 - 194s - 4s/step - accuracy: 0.8992 - auc: 0.9666 - loss: 0.2555 - val_accuracy: 0.9126 - val_auc: 0.9782 - val_loss: 0.2459 - learning_rate: 1.0000e-05


Epoch 3/10


...

44/44 - 176s - 4s/step - accuracy: 0.9618 - auc: 0.9930 - loss: 0.1099 - val_accuracy: 0.9482 - val_auc: 0.9904 - val_loss: 0.1259 - learning_rate: 1.0000e-05


Epoch 10/10


44/44 - 158s - 4s/step - accuracy: 0.9690 - auc: 0.9954 - loss: 0.0920 - val_accuracy: 0.9482 - val_auc: 0.9905 - val_loss: 0.1213 - learning_rate: 1.0000e-05


=== VALIDACIÓN (umbral 0.50) ===
              precision    recall  f1-score   support

      NORMAL      0.954     0.941     0.947       153
   PNEUMONIA      0.943     0.955     0.949       156

    accuracy                          0.948       309
   macro avg      0.948     0.948     0.948       309
weighted avg      0.948     0.948     0.948       309

=== TEST (umbral 0.50) ===
              precision    recall  f1-score   support

      NORMAL      0.946     0.952     0.949       146
   PNEUMONIA      0.955     0.949     0.952       156

    accuracy                          0.950       302
   macro avg      0.950     0.950     0.950       302
weighted avg      0.950     0.950     0.950       302

=== TEST (umbral clínico 0.5810, fijado con validación para recall>=0.95) ===
              precision    recall  f1-score   support

      NORMAL      0.946     0.952     0.949       146
   PNEUMONIA      0.955     0.949     0.952       156

    accuracy                          0.950 

Run 'ref3_finetune_1paso_lr1e-5' -- run_id=de4f0f2972864a5ba3ccbd2a4ec7b77d -- 10 épocas, 30.2 min
  VAL  @0.5: {"accuracy": 0.9482, "precision": 0.943, "recall": 0.9551, "specificity": 0.9412, "f1_score": 0.949, "auc": 0.9907, "tp": 149, "fp": 9, "tn": 144, "fn": 7}
  TEST @0.5: {"accuracy": 0.9503, "precision": 0.9548, "recall": 0.9487, "specificity": 0.9521, "f1_score": 0.9518, "auc": 0.9891, "tp": 148, "fp": 7, "tn": 139, "fn": 8}


## Variante A (propia) — Fine-tuning en **dos fases** real

**Qué cambia:** en vez de empezar con una cabeza aleatoria (Run 3), se parte del **modelo ya
entrenado del Run 1**, se descongela la base desde la capa 143 y se continúa con LR=1e-5.

**Por qué:** es la receta estándar de transfer learning (y la que el propio README describe:
"Fase A ... el punto de partida obligatorio antes de cualquier fine-tuning"), pero el Run 3 no la
aplica. Con una cabeza aleatoria, los primeros gradientes son grandes y ruidosos y llegan a las capas
recién descongeladas: o se usa un LR tan bajo que la cabeza aprende muy lento (Run 3), o se dañan
los filtros preentrenados. Partiendo de una cabeza ya entrenada, el fine-tuning solo tiene que
*refinar*.

**Hipótesis medible:** con el mismo presupuesto de épocas, la Variante A supera al Run 1 (del que
parte) y al Run 3 en `val_loss` y F1 de validación.

In [6]:
run_id_A, model_A, resultados["varA"] = train_and_log(
    train_ds, val_ds, test_ds,
    run_name="varA_finetune_2fases_lr1e-5",
    learning_rate=1e-5, dropout=0.3, epochs=EPOCHS,
    fine_tune=True, fine_tune_at=FINE_TUNE_AT,
    initial_model=model_1, parent_run_id=run_id_1,
    class_weight=class_weight,
    tags={"fase": "B_fine_tuning", "tipo": "variante_propia"},
    justification=("Fine-tuning en dos fases real: parte del modelo del Run 1 (cabeza ya entrenada) "
                   "en lugar de una cabeza aleatoria como el Run 3."),
)

Epoch 1/10


44/44 - 167s - 4s/step - accuracy: 0.9294 - auc: 0.9770 - loss: 0.1942 - val_accuracy: 0.9353 - val_auc: 0.9781 - val_loss: 0.1816 - learning_rate: 1.0000e-05


Epoch 2/10


44/44 - 148s - 3s/step - accuracy: 0.9460 - auc: 0.9849 - loss: 0.1553 - val_accuracy: 0.9288 - val_auc: 0.9795 - val_loss: 0.1801 - learning_rate: 1.0000e-05


Epoch 3/10


...

44/44 - 153s - 3s/step - accuracy: 0.9770 - auc: 0.9966 - loss: 0.0757 - val_accuracy: 0.9482 - val_auc: 0.9909 - val_loss: 0.1211 - learning_rate: 1.0000e-05


Epoch 10/10


44/44 - 198s - 4s/step - accuracy: 0.9741 - auc: 0.9977 - loss: 0.0697 - val_accuracy: 0.9612 - val_auc: 0.9914 - val_loss: 0.1190 - learning_rate: 1.0000e-05


=== VALIDACIÓN (umbral 0.50) ===
              precision    recall  f1-score   support

      NORMAL      0.973     0.948     0.960       153
   PNEUMONIA      0.950     0.974     0.962       156

    accuracy                          0.961       309
   macro avg      0.962     0.961     0.961       309
weighted avg      0.961     0.961     0.961       309

=== TEST (umbral 0.50) ===
              precision    recall  f1-score   support

      NORMAL      0.944     0.918     0.931       146
   PNEUMONIA      0.925     0.949     0.937       156

    accuracy                          0.934       302
   macro avg      0.934     0.933     0.934       302
weighted avg      0.934     0.934     0.934       302

=== TEST (umbral clínico 0.5853, fijado con validación para recall>=0.95) ===
              precision    recall  f1-score   support

      NORMAL      0.938     0.932     0.935       146
   PNEUMONIA      0.936     0.942     0.939       156

    accuracy                          0.937 

Run 'varA_finetune_2fases_lr1e-5' -- run_id=648e23a2957d48a18259a7d879207c38 -- 10 épocas, 30.2 min
  VAL  @0.5: {"accuracy": 0.9612, "precision": 0.95, "recall": 0.9744, "specificity": 0.9477, "f1_score": 0.962, "auc": 0.9914, "tp": 152, "fp": 8, "tn": 145, "fn": 4}
  TEST @0.5: {"accuracy": 0.9338, "precision": 0.925, "recall": 0.9487, "specificity": 0.9178, "f1_score": 0.9367, "auc": 0.9867, "tp": 148, "fp": 12, "tn": 134, "fn": 8}


## Variante B (propia) — Ablación: Run 1 **sin flip horizontal**

**Qué cambia:** exactamente la configuración del Run 1, con una sola diferencia: la augmentation no
incluye `RandomFlip("horizontal")`. Es un experimento controlado: si los resultados cambian, la
causa es el flip.

**Por qué:** en una radiografía de tórax el flip produce una anatomía que casi no existe (corazón a
la derecha) y espeja el marcador lateral "R" de la placa (visible en `02_preprocessing.ipynb`). Puede
ayudar como regularizador o puede enseñar variaciones irreales. Es una decisión de dominio que
conviene **medir** en lugar de suponer.

**Hipótesis medible:** si el flip no aporta, la Variante B iguala o mejora al Run 1 en validación.

In [7]:
run_id_B, _, resultados["varB"] = train_and_log(
    train_ds_sin_flip, val_ds, test_ds,
    run_name="varB_frozen_sin_flip",
    learning_rate=1e-3, dropout=0.3, epochs=EPOCHS,
    fine_tune=False, class_weight=class_weight,
    tags={"fase": "A_congelado", "tipo": "variante_propia"},
    extra_params={"augmentation": "rotation+zoom+contrast (SIN horizontal flip)"},
    justification=("Ablación del Run 1: única diferencia es quitar el flip horizontal, "
                   "que en rayos X de tórax genera anatomía irreal (corazón a la derecha) y espeja el marcador 'R'."),
)

Epoch 1/10


44/44 - 128s - 3s/step - accuracy: 0.8071 - auc: 0.8950 - loss: 0.4095 - val_accuracy: 0.9126 - val_auc: 0.9652 - val_loss: 0.2527 - learning_rate: 0.0010


Epoch 2/10


44/44 - 137s - 3s/step - accuracy: 0.8927 - auc: 0.9553 - loss: 0.2697 - val_accuracy: 0.9094 - val_auc: 0.9720 - val_loss: 0.2185 - learning_rate: 0.0010


Epoch 3/10


...

44/44 - 113s - 3s/step - accuracy: 0.9410 - auc: 0.9840 - loss: 0.1604 - val_accuracy: 0.9191 - val_auc: 0.9800 - val_loss: 0.1972 - learning_rate: 0.0010


Epoch 10/10


44/44 - 147s - 3s/step - accuracy: 0.9338 - auc: 0.9822 - loss: 0.1690 - val_accuracy: 0.9320 - val_auc: 0.9816 - val_loss: 0.1764 - learning_rate: 0.0010


=== VALIDACIÓN (umbral 0.50) ===
              precision    recall  f1-score   support

      NORMAL      0.940     0.922     0.931       153
   PNEUMONIA      0.925     0.942     0.933       156

    accuracy                          0.932       309
   macro avg      0.932     0.932     0.932       309
weighted avg      0.932     0.932     0.932       309

=== TEST (umbral 0.50) ===
              precision    recall  f1-score   support

      NORMAL      0.911     0.911     0.911       146
   PNEUMONIA      0.917     0.917     0.917       156

    accuracy                          0.914       302
   macro avg      0.914     0.914     0.914       302
weighted avg      0.914     0.914     0.914       302

=== TEST (umbral clínico 0.3539, fijado con validación para recall>=0.95) ===
              precision    recall  f1-score   support

      NORMAL      0.934     0.877     0.905       146
   PNEUMONIA      0.891     0.942     0.916       156

    accuracy                          0.911 

Run 'varB_frozen_sin_flip' -- run_id=bd8539c8a4134903a0f3fb42ccdeb826 -- 10 épocas, 21.5 min
  VAL  @0.5: {"accuracy": 0.932, "precision": 0.9245, "recall": 0.9423, "specificity": 0.9216, "f1_score": 0.9333, "auc": 0.9812, "tp": 147, "fp": 12, "tn": 141, "fn": 9}
  TEST @0.5: {"accuracy": 0.9139, "precision": 0.9167, "recall": 0.9167, "specificity": 0.911, "f1_score": 0.9167, "auc": 0.9787, "tp": 143, "fp": 13, "tn": 133, "fn": 13}


## Resumen de los 5 runs

In [8]:
filas = []
for nombre, r in resultados.items():
    filas.append({
        "run": nombre,
        "val_f1": r["val"]["f1_score"], "val_recall": r["val"]["recall"], "val_auc": r["val"]["auc"],
        "test_acc": r["test"]["accuracy"], "test_f1": r["test"]["f1_score"],
        "test_recall": r["test"]["recall"], "test_specificity": r["test"]["specificity"], "test_auc": r["test"]["auc"],
    })
resumen = pd.DataFrame(filas).set_index("run").round(4)
resumen

,val_f1,val_recall,val_auc,test_acc,test_f1,test_recall,test_specificity,test_auc
run,,,,,,,,
ref1,0.9367,0.9487,0.9823,0.9139,0.9161,0.9103,0.9178,0.9792
ref2,0.9091,0.8974,0.9669,0.8874,0.8889,0.8718,0.9041,0.9632
ref3,0.9490,0.9551,0.9907,0.9503,0.9518,0.9487,0.9521,0.9891
varA,0.9620,0.9744,0.9914,0.9338,0.9367,0.9487,0.9178,0.9867
varB,0.9333,0.9423,0.9812,0.9139,0.9167,0.9167,0.9110,0.9787


In [9]:
run_ids = {"ref1": run_id_1, "ref2": run_id_2, "ref3": run_id_3, "varA": run_id_A, "varB": run_id_B}
print("Runs completados:", run_ids)

Runs completados: {'ref1': 'a4bf9a5c2e6b454d8e34a022562e6aef', 'ref2': '284800fdc5ed41ad8769128820a038ed', 'ref3': 'de4f0f2972864a5ba3ccbd2a4ec7b77d', 'varA': '648e23a2957d48a18259a7d879207c38', 'varB': 'bd8539c8a4134903a0f3fb42ccdeb826'}


Siguiente: `04_evaluation.ipynb` (comparación, selección, Model Registry y export).